In [1]:
import duckdb
from pathlib import Path
import plotly.express as px

import polars as pl


In [2]:
LAND_REGISTRY_DATABASE_PATH = Path("../../databases/land_registry.db")

In [3]:
db = duckdb.connect(LAND_REGISTRY_DATABASE_PATH, read_only=True)

# High level overview of data

Finally we convert the Gold `average_price_by_month_and_property_type` view to a Polars dataframe so we can visualise the data on a line chart using the Plotly Express charting package.

In [4]:

fig = px.line(
    db.sql("FROM gold.median_price_by_month_and_property_type").pl(),
    x="month_of_sale",
    y="median_price",
    color="property_type",
    line_dash="property_type",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    line_dash_sequence=["solid", "dash", "dot", "dashdot"],
    title="Median Price by Month and Property Type"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Median Price",
    legend_title="Property Type",
)

fig.show()

In [7]:
def generate_insights_for_postcode(
    postcode: str,
    property_type: str | None = None,
    min_date: str | None = None,
    max_date: str | None = None,
) -> duckdb.DuckDBPyRelation:

    query = db.sql("SELECT * FROM silver.price_paid_cleaned")

    query = db.sql(f"SELECT * FROM query WHERE postcode LIKE '{postcode}%'")

    if property_type:
        query = db.sql(f"SELECT * FROM query WHERE property_type = '{property_type}'")

    if min_date:
        query = db.sql(f"SELECT * FROM query WHERE date >= '{min_date}'")

    if max_date:
        query = db.sql(f"SELECT * FROM query WHERE date <= '{max_date}'")

    query = db.sql("""
        SELECT
            property_type,
            date_trunc('month', date) AS month_of_sale,
            MEDIAN(price) AS median_price,
            COUNT(*) AS number_of_sales
        FROM query
        GROUP BY ALL
        ORDER BY month_of_sale, property_type
    """)

    return query

In [16]:
generate_insights_for_postcode("NP19", property_type="Terraced").pl()

property_type,month_of_sale,median_price,number_of_sales
str,datetime[μs],f64,i64
"""Terraced""",2023-01-01 00:00:00,168997.5,36
"""Terraced""",2023-02-01 00:00:00,160000.0,46
"""Terraced""",2023-03-01 00:00:00,170500.0,40
"""Terraced""",2023-04-01 00:00:00,140000.0,43
"""Terraced""",2023-05-01 00:00:00,158000.0,29
…,…,…,…
"""Terraced""",2025-11-01 00:00:00,174000.0,24
"""Terraced""",2025-12-01 00:00:00,170000.0,36
"""Terraced""",2026-01-01 00:00:00,175000.0,21
